# Group 5 Project 4 - Algorithmic Bias

## Section 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
import matplotlib.pyplot as plt

In [3]:
import seaborn as sns

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
from sklearn.preprocessing import StandardScaler

In [6]:
from sklearn.linear_model import LogisticRegression

In [7]:
from sklearn.metrics import confusion_matrix, classification_report

In [9]:
pd.set_option('display.max_columns', 500)

## Section 2. Import Data



In [10]:
df_covenant_mortgage = pd.read_csv('DATA/mortgage_covenant_data.csv', index_col=0, header=0)

In [11]:
df_covenant_mortgage.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100931 entries, 2023 to 2023
Data columns (total 13 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   census_tract                       99915 non-null   float64
 1   derived_race                       100931 non-null  object 
 2   action_taken                       100931 non-null  int64  
 3   loan_amount                        100931 non-null  float64
 4   property_value                     83491 non-null   float64
 5   income                             88200 non-null   float64
 6   interest_rate                      78125 non-null   float64
 7   tract_minority_population_percent  100931 non-null  float64
 8   tract_to_msa_income_percentage     100931 non-null  float64
 9   denial_reason-1                    100931 non-null  int64  
 10  covenant_count                     100931 non-null  float64
 11  was_approved                       100931 n

In [18]:
df_covenant_mortgage.shape

(100931, 13)

In [20]:
df_covenant_mortgage.columns

Index(['census_tract', 'derived_race', 'action_taken', 'loan_amount',
       'property_value', 'income', 'interest_rate',
       'tract_minority_population_percent', 'tract_to_msa_income_percentage',
       'denial_reason-1', 'covenant_count', 'was_approved',
       'covenant_density'],
      dtype='object')

In [23]:
df_covenant_mortgage.head()

,census_tract,derived_race,action_taken,loan_amount,property_value,income,interest_rate,tract_minority_population_percent,tract_to_msa_income_percentage,denial_reason-1,covenant_count,was_approved,covenant_density
activity_year,,,,,,,,,,,,,
2023,2.703706e+10,Race Not Available,6,185000.0,195000.0,NaN,6.125,40.37,73.48,10,0.0,False,NaN
2023,2.714103e+10,Race Not Available,6,375000.0,385000.0,NaN,6.625,10.64,121.26,10,0.0,False,NaN
2023,2.702395e+10,Race Not Available,6,105000.0,105000.0,NaN,6.125,8.92,90.13,10,0.0,False,NaN
2023,2.717110e+10,Race Not Available,6,285000.0,285000.0,NaN,6.750,13.27,104.37,10,0.0,False,NaN
2023,2.715948e+10,Race Not Available,6,175000.0,175000.0,NaN,6.875,5.87,82.22,10,0.0,False,NaN


## Initial Analysis 

In [24]:
# Clean Covenant Variables
# Fill missing covenant_count
df_covenant_mortgage["covenant_count"] = df_covenant_mortgage["covenant_count"].fillna(0)

# Create historical indicator
df_covenant_mortgage["historically_covenanted"] = df_covenant_mortgage["covenant_count"] > 0

# Replace missing covenant_density with 0 (non-covenanted)
df_covenant_mortgage["covenant_density"] = df_covenant_mortgage["covenant_density"].fillna("0")

df_covenant_mortgage["covenant_density"].value_counts()

covenant_density
0                   89762
Low (1-50)           4537
Very High (>200)     3140
Medium (51-100)      1772
High (101-200)       1720
Name: count, dtype: int64

In [25]:
#Restrict to application only by removing purchased loans
apps = df_covenant_mortgage[df_covenant_mortgage["action_taken"].isin([1,2,3,4,5])].copy()

print("Applications:", apps.shape)
print("Approval rate:", apps["was_approved"].mean())

Applications: (84709, 14)
Approval rate: 0.7919465464118335


In [26]:
apps.groupby("historically_covenanted")["property_value"].mean()

#saw difference of about $106K for property value difference of historically covenanted

historically_covenanted
False    397896.548276
True     504175.120128
Name: property_value, dtype: float64

In [27]:
apps.groupby("historically_covenanted")["property_value"].median()

#Covenanted tracts have about $40K higher median home values

historically_covenanted
False    325000.0
True     365000.0
Name: property_value, dtype: float64

In [28]:
apps.groupby("historically_covenanted")["was_approved"].mean()

historically_covenanted
False    0.789805
True     0.809395
Name: was_approved, dtype: float64

In [29]:
apps.groupby("historically_covenanted")[
    ["tract_minority_population_percent",
     "tract_to_msa_income_percentage",
     "income",
     "property_value"]
].mean()

,tract_minority_population_percent,tract_to_msa_income_percentage,income,property_value
historically_covenanted,,,,
False,21.712257,101.176134,151.445817,397896.548276
True,29.914679,111.771434,167.095076,504175.120128


Areas that are historically covenants now have higher property values. 
* historically_covenanted
    * False    0.789805
    * True     0.809395
        * Raw approval rates are slightly higher in covenanted tracts.
        * Misleading because these areas have higher income, property values, and income percentage.
* Areas that once excluded minorities now have higher minority representation.
* Covenanted neighborhoods are wealthier than average, strong evidence of long-term structural wealth effects.
* Applicants in covenantes tracts earn more. Higher income means higher approval probability.
* Homes are far more expensive in covenanted tracts, higher property values lead to higher loan approval rates.
* Historically covenanted areas are currently wealthier, have higher home values, and applicants there earn more.

## Bias Investigation 

In [30]:
# Approval rate by covenant_density category
approval_by_density = apps.groupby("covenant_density")["was_approved"].agg(["count","mean"]).sort_values("mean", ascending=False)
approval_by_density = approval_by_density.rename(columns={"count":"n", "mean":"approval_rate"})
display(approval_by_density)

,n,approval_rate
covenant_density,,
High (101-200),1451,0.820813
Very High (>200),2605,0.812284
Medium (51-100),1467,0.809816
Low (1-50),3737,0.802783
0,75449,0.789805


In [31]:
# Interest rate differences (loan terms) by covenanted vs non-covenanted
rate_by_covenant = apps.groupby("historically_covenanted")["interest_rate"].agg(["count","mean","median"])
display(rate_by_covenant)

,count,mean,median
historically_covenanted,,,
False,55078,6.564464,6.625
True,7095,6.599989,6.625


In [32]:
rate_by_density = apps.groupby("covenant_density")["interest_rate"].agg(["count","mean","median"])
display(rate_by_density)

,count,mean,median
covenant_density,,,
0,55078,6.564464,6.625
High (101-200),1127,6.622981,6.625
Low (1-50),2820,6.562372,6.750
Medium (51-100),1128,6.609170,6.750
Very High (>200),2020,6.634548,6.625


In [33]:
apps["ltv_proxy"] = apps["loan_amount"] / apps["property_value"]

ltv_summary = apps.groupby("historically_covenanted")["ltv_proxy"].agg(["count","mean","median"])
display(ltv_summary)

,count,mean,median
historically_covenanted,,,
False,59999,0.793984,0.876923
True,7492,0.777731,0.822222


In [34]:
denied = apps[apps["action_taken"] == 3].copy()

denial_counts = (
    denied.groupby(["historically_covenanted","denial_reason-1"])
    .size()
    .reset_index(name="n")
)

denial_counts["share_within_group"] = denial_counts["n"] / denial_counts.groupby("historically_covenanted")["n"].transform("sum")

denial_pivot = denial_counts.pivot_table(
    index="denial_reason-1",
    columns="historically_covenanted",
    values="share_within_group",
    fill_value=0
).rename(columns={False:"Non-covenanted", True:"Covenanted"}).sort_values("Non-covenanted", ascending=False)

display((denial_pivot * 100).round(1))

historically_covenanted,Non-covenanted,Covenanted
denial_reason-1,,
1,37.5,40.1
3,21.0,10.0
4,13.0,17.5
9,7.9,10.0
7,6.0,6.2
5,4.8,3.2
6,4.4,6.0
2,2.8,5.7
1111,2.6,1.2


In [39]:
apps_model["was_approved"] = apps_model["was_approved"].astype(int)

In [40]:

import statsmodels.formula.api as smf
from statsmodels.genmod.families import Binomial

apps_model = apps.dropna(subset=["income","loan_amount","property_value"]).copy()
apps_model = apps_model[
    (apps_model["income"] > 0) &
    (apps_model["loan_amount"] > 0) &
    (apps_model["property_value"] > 0)
]

glm = smf.glm(
    "was_approved ~ historically_covenanted"
    " + np.log(income) + np.log(loan_amount) + np.log(property_value)"
    " + tract_minority_population_percent + tract_to_msa_income_percentage",
    data=apps_model,
    family=Binomial()
).fit(cov_type="HC3")

print(glm.summary())

# Odds ratios for easier interpretation
odds = np.exp(glm.params)
ci = np.exp(glm.conf_int())
or_table = pd.DataFrame({"odds_ratio": odds, "ci_low": ci[0], "ci_high": ci[1], "p_value": glm.pvalues})
display(or_table.loc[["historically_covenanted[T.True]"]])

                               Generalized Linear Model Regression Results                               
Dep. Variable:     ['was_approved[False]', 'was_approved[True]']   No. Observations:                65968
Model:                                                       GLM   Df Residuals:                    65961
Model Family:                                           Binomial   Df Model:                            6
Link Function:                                             Logit   Scale:                          1.0000
Method:                                                     IRLS   Log-Likelihood:                -15916.
Date:                                           Mon, 02 Mar 2026   Deviance:                       31833.
Time:                                                   18:43:14   Pearson chi2:                 9.43e+04
No. Iterations:                                                6   Pseudo R-squ. (CS):            0.05219
Covariance Type:                              

,odds_ratio,ci_low,ci_high,p_value
historically_covenanted[T.True],0.857927,0.762712,0.965029,0.010679


In [41]:
glm2 = smf.glm(
    "was_approved ~ C(covenant_density)"
    " + np.log(income) + np.log(loan_amount) + np.log(property_value)"
    " + tract_minority_population_percent + tract_to_msa_income_percentage",
    data=apps_model,
    family=Binomial()
).fit(cov_type="HC3")

odds2 = np.exp(glm2.params)
ci2 = np.exp(glm2.conf_int())
or_table2 = pd.DataFrame({"odds_ratio": odds2, "ci_low": ci2[0], "ci_high": ci2[1], "p_value": glm2.pvalues})

display(or_table2[or_table2.index.str.contains("C\\(covenant_density\\)")])

,odds_ratio,ci_low,ci_high,p_value
C(covenant_density)[T.High (101-200)],0.692650,0.499227,0.961014,0.027951
C(covenant_density)[T.Low (1-50)],0.810900,0.680159,0.966772,0.019456
C(covenant_density)[T.Medium (51-100)],1.047336,0.805791,1.361286,0.729534
C(covenant_density)[T.Very High (>200)],0.917801,0.743574,1.132852,0.424523


In [42]:
or_val = np.exp(glm.params["historically_covenanted[T.True]"])
ci = np.exp(glm.conf_int().loc["historically_covenanted[T.True]"])
print("Odds Ratio:", round(or_val, 3), " CI:", tuple(round(x,3) for x in ci))

Odds Ratio: 0.858  CI: (0.763, 0.965)


After controlling for applicant income, loan amount, property value, and neighborhood characteristics, applications in historically covenanted tracts have approximately 14% lower odds of approval compared to non-covenanted tracts (OR = 0.858, 95% CI: 0.763–0.965, p = 0.011).

Although covenanted tracts exhibit higher average property values and incomes, the controlled model suggests that approval outcomes are not fully explained by observable financial factors.

## Ramsey County Extension